## include_raw=True 参数

添加 `include_raw=True` 参数后，如果 parse 阶段的 pydantic 校验没有通过，不会直接抛出异常，而是在 response dictionary 中的 `parsing_error` 中填充异常信息

In [18]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from pydantic import SecretStr

# 从.env文件中加载环境变量
load_dotenv(override=True)
model = init_chat_model(model="openai:openai/gpt-5.6-luna", max_tokens=64890, base_url="http://localhost:8889",
                        api_key=SecretStr("<KEY>"))

from pydantic import BaseModel, Field
from rich import print as rprint


class Movie(BaseModel):
    """电影信息"""
    title: str = Field(description="电影标题")
    year: int = Field(description="上映年份")
    director: str = Field(description="导演")
    rating: float = Field(description="评分（10分制）")


# 设置模型结构化输出
model_with_structure = model.with_structured_output(Movie, method="function_calling", include_raw=True)
# 调用模型并获取结构化输出

resp = model_with_structure.invoke("给我介绍下电影《星际穿越》")

print(type(resp))
rprint(resp)
rprint(resp.get("parsed"))

<class 'dict'>


{
    'raw': AIMessage(
        content='',
        additional_kwargs={'refusal': None},
        response_metadata={
            'token_usage': {
                'completion_tokens': 1,
                'prompt_tokens': 1,
                'total_tokens': 2,
                'completion_tokens_details': None,
                'prompt_tokens_details': None
            },
            'model_provider': 'openai',
            'model_name': 'any',
            'system_fingerprint': None,
            'id': 'chatcmpl-test',
            'finish_reason': 'stop',
            'logprobs': None
        },
        id='lc_run--019f9d09-6e45-7be2-a20c-e5574955a4e6-0',
        tool_calls=[
            {
                'name': 'Movie',
                'args': {
                    'title1': '盗梦空间',
                    'year2': 2010,
                    'year': 2010,
                    'director': '克里斯托弗·诺兰',
                    'rating': 9.3
                },
                'id': 'call_1',
                'type': 'tool_call'
            }
        ],
        invalid_tool_calls=[],
        usage_metadata={
            'input_tokens': 1,
            'output_tokens': 1,
            'total_tokens': 2,
            'input_token_details': {},
            'output_token_details': {}
        }
    ),
    'parsing_error': 1 validation error for Movie
title
  Field required [type=missing, input_value={'title1': '盗梦空间'...诺兰', 'rating': 9.3}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing,
    'parsed': None
}

None